# 02 — KV-cache reuse: vLLM prefix caching and LMCache

Every request to the agent starts with the same system prompt and 13 tool
schemas, and each step of a tool-calling turn repeats everything before it. A
server that keeps the KV cache of text it has already processed can skip that
work, which shows up as a lower **time-to-first-token (TTFT)**.

Three setups, each on a freshly started server, replaying the same fixed trace
of multi-step tool-calling conversations (`bench/trace.jsonl`):

| Setup | What it shows |
|---|---|
| A. vLLM, prefix caching off | Baseline: every prompt is fully recomputed |
| B. vLLM (default) | Prefix reuse from GPU memory |
| C. vLLM + LMCache | Prefix reuse from GPU memory, CPU RAM, and local disk |

Then an **eviction test** on B and C: a conversation is sent, pushed out of GPU
memory by unrelated prompts, and sent again.

The GPU KV cache is capped at 2 GB in all three setups so that eviction happens
within a short test. That cap is the only non-default setting.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: vLLM + LMCache in their own virtualenv ---
# The server runs as a separate process, so it gets a separate environment.
# That keeps its packages from clashing with the ones Colab ships in the notebook.
VENV = "/content/venv-vllm"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python vllm lmcache
os.environ["SERVING_VENV"] = VENV

# Record the exact versions so the results can be reproduced.
!uv pip freeze --python {VENV}/bin/python | grep -iE "^(vllm|lmcache|torch|transformers)==" | tee results/versions_vllm.txt

In [ ]:
from serving.launch import start_server, stop_server

MODEL = "Qwen/Qwen3-8B-AWQ"
KV_CAP = "--kv-cache-memory-bytes 2147483648"   # 2 GB GPU KV cache, same for every setup
EVICTION = "--targets 3 --fillers 12 --filler-words 2500"

## A. vLLM with prefix caching off

In [ ]:
server = start_server("vllm", MODEL, extra_args=f"--no-enable-prefix-caching {KV_CAP}", log_name="kv_no_cache")
!python -m bench.ttft --name ttft_no_cache --model {MODEL}
stop_server(server)

## B. vLLM with prefix caching (default)

In [ ]:
server = start_server("vllm", MODEL, extra_args=KV_CAP, log_name="kv_prefix_cache")
!python -m bench.ttft --name ttft_prefix_cache --model {MODEL}
!python -m bench.eviction --name evict_prefix_cache --model {MODEL} {EVICTION}
!curl -s localhost:8000/metrics | grep -E "prefix_cache|kv_cache|cache_usage" | grep -v "^#" > results/ttft_prefix_cache/metrics.txt; cat results/ttft_prefix_cache/metrics.txt
stop_server(server)

## C. vLLM + LMCache (CPU RAM + local disk)

In [ ]:
!rm -rf /content/lmcache_disk
server = start_server("vllm_lmcache", MODEL, extra_args=KV_CAP, log_name="kv_lmcache")
!python -m bench.ttft --name ttft_lmcache --model {MODEL}
!python -m bench.eviction --name evict_lmcache --model {MODEL} {EVICTION}
!curl -s localhost:8000/metrics | grep -E "prefix_cache|kv_cache|cache_usage" | grep -v "^#" > results/ttft_lmcache/metrics.txt; cat results/ttft_lmcache/metrics.txt
stop_server(server)

# Evidence that LMCache was used: its log lines, and how much KV cache reached the disk tier
!grep -iE "lmcache" results/logs/kv_lmcache.log | grep -iE "stor|retriev|load|hit" | tail -n 30
!du -sh /content/lmcache_disk | tee results/ttft_lmcache/disk_usage.txt

## Results

In [ ]:
import pandas as pd

display(pd.read_csv("results/ttft_summary.csv"))
display(pd.read_csv("results/eviction_summary.csv"))

In [ ]:
# --- Download the results ---
!zip -qr /content/results_02_kv_cache.zip results
from google.colab import files
files.download("/content/results_02_kv_cache.zip")